# 04 · Estabilidad pareada, lista objetivo y tratamiento

Declara la regla y los métodos de corte antes de calcular, solo usa Y1–Y4 (FR-021) y congela la lista objetivo con su tratamiento y los parámetros de segmentación (FR-020). El CV usa las rachas de ceros en NaN según `zero_thresholds` (etapa 3).

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

Estabilidad pareada (FR-016–FR-018, R14): cada año de train se compara con el año anterior (`prev`) y con Y1 (`y1`), con los mismos artículos (activos todos los días de los dos años). Los cortes se eligen en la base (regla ABC y `find_cv_cuts`) y el objetivo se clasifica ABC por participación en el gasto. También se compara la matriz global de train con cada año (`global`) y se mide una línea base de ruido (`noise`: semanas alternas para el gasto y bloques de 28 días alternos para el CV, dentro del mismo año).

**Límites:** con miles de artículos Stuart–Maxwell rechaza ante cambios pequeños; leerlo junto con `max_shift_pp`. El kappa es plano: un salto A → C pesa lo mismo que A → B.

In [ ]:
from src.stability import run_stability_stage

res = run_stability_stage(cfg)
measures = res["measures"]
cols = ["n", "kappa", "stuart_maxwell", "sm_pvalue", "max_shift_pp", "max_shift_class",
        "keep_pct", "up_pct", "down_pct", "mixed_pct"]
fmt = {"kappa": "{:.2f}", "stuart_maxwell": "{:.1f}", "sm_pvalue": "{:.3g}", "max_shift_pp": "{:.1f}",
       "keep_pct": "{:.1f}", "up_pct": "{:.1f}", "down_pct": "{:.1f}", "mixed_pct": "{:.1f}"}
for level in ("abc", "xyz", "cell"):
    t = measures[measures["level"] == level].set_index(["base_kind", "base", "target"])[cols]
    display(t.style.format(fmt, na_rep="–").set_caption(f"Nivel {level}"))
notes = measures.dropna(subset=["note"])
if len(notes):
    display(notes[["base_kind", "base", "target", "level", "note"]])

Matrices de transición (% de la fila) de las comparaciones con el año anterior, en los tres niveles.

In [ ]:
import matplotlib.pyplot as plt

from src.stability import TRANSITION_LEVELS

prev = [k for k in res["comparisons"] if k[0] == "prev"]
for level, (labels, _) in TRANSITION_LEVELS.items():
    fig, axes = plt.subplots(1, len(prev), figsize=(4.2 * len(prev), 3.8 if level != "cell" else 5))
    for ax, key in zip(np.atleast_1d(axes), prev):
        it = res["comparisons"][key]["items"]
        it = it[(it["status"] == "both") & it[f"ref_{level}"].notna() & it[f"tgt_{level}"].notna()]
        ct = pd.crosstab(it[f"ref_{level}"], it[f"tgt_{level}"]).reindex(index=labels, columns=labels, fill_value=0)
        pct = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0) * 100
        ax.imshow(pct.fillna(0), cmap="Blues", vmin=0, vmax=100)
        ax.set(xticks=range(len(labels)), yticks=range(len(labels)), xticklabels=labels, yticklabels=labels,
               title=f"{key[1]} → {key[2]}", xlabel="objetivo", ylabel="base")
        for r in range(len(labels)):
            for q in range(len(labels)):
                if ct.iat[r, q]:
                    ax.text(q, r, f"{pct.iat[r, q]:.0f}", ha="center", va="center", fontsize=7)
    fig.suptitle(f"Transiciones · {level}")
    plt.tight_layout()
    plt.show()

Cambios por artículo en las comparaciones con el año anterior (la regla `stability_rule` se aplica sobre estas).

In [ ]:
st = res["item_stability"]
print("regla:", st["rule"].iloc[0])
st.groupby("comparison")[["same_abc", "same_xyz", "same_cell"]].mean()

Lista objetivo congelada: artículos A estables en train con su tratamiento.

In [ ]:
targets = res["targets"]
print(len(targets), "productos objetivo")
display(targets.groupby(["cell", "treatment"]).size())
targets.head(20)